In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def churn_logistic_model(df, feature_cols, target_col):
    """
    Fit a logistic regression to predict churn.
    Standardize features first so coefficients are comparable.

    Parameters:
        df          : DataFrame with features and target
        feature_cols: list of feature column names
        target_col  : binary target column (1 = churned, 0 = retained)

    Returns a dict with:
        - feature_ranking : DataFrame with columns
                            ['feature', 'coefficient', 'odds_ratio']
                            sorted by abs(coefficient) descending
        - top_churn_driver: name of single most important feature
        - auc             : ROC-AUC on training data
        - n_obs           : number of observations used
        - churn_rate      : baseline churn rate in the dataset
    """
    # ── Input validation ──────────────────────────────────
    assert len(feature_cols) > 0, "feature_cols must not be empty"
    assert target_col in df.columns, f"{target_col} not in DataFrame"
    assert all(c in df.columns for c in feature_cols), "some feature_cols missing"
    assert df[target_col].nunique() == 2, "target must be binary (0/1)"

    # ── Step 1: Standardize features ─────────────────────
    # Fit on the data and transform — coefficients now in SD units
    # so they are directly comparable across features
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feature_cols])   # shape: (n_obs, n_features)
    y = df[target_col].values

    # ── Step 2: Fit logistic regression ──────────────────
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X, y)

    # ── Step 3: Extract coefficients + odds ratios ───────
    # coef_[0] because sklearn returns shape (1, n_features) for binary
    coefficients = logreg.coef_[0]
    odds_ratios  = np.exp(coefficients)

    # ── Step 4: Build ranking DataFrame ──────────────────
    # Sort by absolute coefficient so both positive and negative
    # drivers are ranked correctly
    feature_ranking = pd.DataFrame({
        'feature'    : feature_cols,
        'coefficient': coefficients,
        'odds_ratio' : odds_ratios
    }).sort_values('coefficient', key=abs, ascending=False)\
      .reset_index(drop=True)

    # ── Step 5: Remaining outputs ─────────────────────────
    top_churn_driver = feature_ranking.iloc[0]['feature']

    # predict_proba returns (n, 2) — column 1 = P(churn=1)
    auc = roc_auc_score(y, logreg.predict_proba(X)[:, 1])

    n_obs      = len(df)
    churn_rate = y.mean()

    # ── Output validation ─────────────────────────────────
    assert 0 <= auc <= 1
    assert len(feature_ranking) == len(feature_cols)

    return {
        'feature_ranking' : feature_ranking,
        'top_churn_driver': top_churn_driver,
        'auc'             : round(auc, 4),
        'n_obs'           : n_obs,
        'churn_rate'      : churn_rate
    }


